# LeetCode 1904 — The Number of Full Rounds You Have Played bằng Python

Link đề: https://leetcode.com/problems/the-number-of-full-rounds-you-have-played/ · Độ khó: **Medium** · Chủ đề: Math, String

Skeleton LeetCode:

```python
class Solution:
    def numberOfRounds(self, loginTime: str, logoutTime: str) -> int:
        ...
```

Lộ trình:
1. **Mô phỏng từng phút**: đi từ lúc đăng nhập tới lúc đăng xuất (vòng qua nửa đêm nếu cần), đếm các ván bắt đầu và kết thúc trọn vẹn trong khoảng (`O(1440)`).
2. **Liệt kê 96 ván mỗi ngày** (và ván của "ngày hôm sau" khi qua nửa đêm), kiểm tra từng ván có nằm trọn trong khoảng chơi không.
3. **Công thức làm tròn**: ván đầu tiên trọn vẹn bắt đầu ở `⌈login / 15⌉`, ván cuối kết thúc ở `⌊logout / 15⌋` ⇒ đáp án `max(0, ⌊out/15⌋ − ⌈in/15⌉)` — sau khi cộng `1440` vào `logout` nếu qua nửa đêm.
4. Biến thể viết bằng **số nguyên thuần** (`−(−a // b)` cho trần) và xử lý trường hợp **cùng một ô 15 phút**.

## 1. Mục tiêu học

1. Chuyển chuỗi giờ `"hh:mm"` thành **số phút từ nửa đêm** — biểu diễn giúp so sánh và tính toán dễ dàng.
2. Xử lý khoảng thời gian **vòng qua nửa đêm** bằng cách cộng `24·60` vào thời điểm kết thúc.
3. Đếm số bội của 15 trong một đoạn bằng **làm tròn lên / làm tròn xuống** (`⌈⌉`, `⌊⌋`) và hiểu vì sao cần `max(0, …)`.
4. Kiểm chứng công thức bằng **vét cạn mọi cặp thời điểm** (1440 × 1440 ≈ 2 triệu cặp có thể chạy hết!).
5. Tránh các lỗi làm tròn với số âm và lỗi "đăng nhập đúng lúc ván bắt đầu".

## 2. Đề bài & ví dụ

Bạn tham gia một giải cờ vua trực tuyến. Cứ **15 phút** lại có một ván bắt đầu; ván đầu tiên trong ngày bắt đầu lúc `00:00`. Ví dụ ván thứ hai bắt đầu `00:15`, ván thứ tư `00:45`, ván thứ bảy `01:30`.

Cho hai chuỗi `loginTime` (lúc đăng nhập) và `logoutTime` (lúc đăng xuất). Nếu `logoutTime` **sớm hơn** `loginTime`, nghĩa là bạn chơi từ `loginTime` tới nửa đêm rồi từ nửa đêm tới `logoutTime`.

Trả về số **ván trọn vẹn** bạn đã chơi (ván bắt đầu không sớm hơn lúc đăng nhập và kết thúc không muộn hơn lúc đăng xuất).

Mọi thời điểm theo giờ 24h: ván đầu ngày bắt đầu `00:00`, ván cuối ngày bắt đầu `23:45`.

**Ví dụ 1:** `"09:31"`, `"10:14"` → `1` (chỉ ván `09:45–10:00`; ván `09:30–09:45` đã bắt đầu trước khi vào; ván `10:00–10:15` chưa xong khi ra).

**Ví dụ 2:** `"21:30"`, `"03:00"` → `22` (10 ván `21:30 → 00:00` + 12 ván `00:00 → 03:00`).

**Ràng buộc:** định dạng `hh:mm`, `00 ≤ hh ≤ 23`, `00 ≤ mm ≤ 59`, `loginTime ≠ logoutTime`.

In [1]:
import random
import time
from typing import List

examples = [(("09:31", "10:14"), 1), (("21:30", "03:00"), 22)]
for (a, b), want in examples:
    print(f"login={a}, logout={b} -> kỳ vọng {want}")

login=09:31, logout=10:14 -> kỳ vọng 1
login=21:30, logout=03:00 -> kỳ vọng 22


## 3. Kiến thức nền

### 3.1 Chuỗi giờ → số phút

In [2]:
def to_min(t):
    h, m = t.split(":")
    return int(h) * 60 + int(m)

print(to_min("00:00"), to_min("09:31"), to_min("23:59"), "| hoặc:", int("21:30"[:2]) * 60 + int("21:30"[3:]))

0 571 1439 | hoặc: 1290


### 3.2 Vòng qua nửa đêm

Nếu `out < in` (theo phút), khoảng chơi là `[in, out + 1440]` trên trục thời gian kéo dài sang ngày hôm sau. Ván bắt đầu lúc `00:00` của ngày sau ứng với phút `1440`.

In [3]:
a, b = to_min("21:30"), to_min("03:00")
print(a, b, "->", (a, b + 1440) if b < a else (a, b))

1290 180 -> (1290, 1620)


### 3.3 Làm tròn lên và xuống với số nguyên

- `⌊a / 15⌋` = `a // 15`.
- `⌈a / 15⌉` = `(a + 14) // 15` = `-(-a // 15)` (với `a ≥ 0`). Cách `-(-a // b)` đúng cả với số âm.

In [4]:
for a in [0, 1, 14, 15, 16, 30, -1, -16]:
    print(f"a={a:4d}: floor = {a // 15:3d}, ceil = {-(-a // 15):3d}, (a+14)//15 = {(a + 14) // 15:3d}")

a=   0: floor =   0, ceil =   0, (a+14)//15 =   0
a=   1: floor =   0, ceil =   1, (a+14)//15 =   1
a=  14: floor =   0, ceil =   1, (a+14)//15 =   1
a=  15: floor =   1, ceil =   1, (a+14)//15 =   1
a=  16: floor =   1, ceil =   2, (a+14)//15 =   2
a=  30: floor =   2, ceil =   2, (a+14)//15 =   2
a=  -1: floor =  -1, ceil =   0, (a+14)//15 =   0
a= -16: floor =  -2, ceil =  -1, (a+14)//15 =  -1


Với mẫu số dương, cả hai cách tính trần đều đúng (kể cả khi `a` âm — bảng trên). Cách `-(-a // b)` tiện hơn khi `b` là biến vì không phải nhớ hằng số `b − 1`.

### 3.4 Đếm bội số trong đoạn

Số bội của `15` trong đoạn số nguyên `[L, R]` là `⌊R/15⌋ − ⌈L/15⌉ + 1` (nếu không âm). Ở bài này ta đếm **ván**, tức số cặp "bắt đầu `s`, kết thúc `s + 15`" nằm trong `[in, out]` ⇒ `s ∈ [in, out − 15]` là bội của 15 ⇒ `⌊(out − 15)/15⌋ − ⌈in/15⌉ + 1 = ⌊out/15⌋ − ⌈in/15⌉`.

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `in = 9·60 + 31 = 571`, `out = 10·60 + 14 = 614`.
- Ván đầu tiên có thể chơi trọn: bắt đầu ở bội của 15 đầu tiên `≥ 571` ⇒ `585` (`09:45`) ⇒ chỉ số ván `⌈571/15⌉ = 39`.
- Ván cuối cùng phải **kết thúc** `≤ 614` ⇒ mốc kết thúc là bội của 15 lớn nhất `≤ 614` ⇒ `600` (`10:00`) ⇒ `⌊614/15⌋ = 40`.
- Số ván = số khoảng giữa hai mốc `39 → 40` ⇒ `40 − 39 = 1`. ✓

Ví dụ 2: `in = 1290`, `out = 180 + 1440 = 1620`. `⌈1290/15⌉ = 86`, `⌊1620/15⌋ = 108` ⇒ `22`. ✓

**Trường hợp đặc biệt:** `in = 10:01`, `out = 10:14` (cùng một ô 15 phút, không chơi trọn ván nào): `⌈601/15⌉ = 41`, `⌊614/15⌋ = 40` ⇒ hiệu `−1` ⇒ phải lấy `max(0, …)`.

In [5]:
for a, b in [("09:31", "10:14"), ("21:30", "03:00"), ("10:01", "10:14")]:
    i, o = to_min(a), to_min(b)
    if o < i: o += 1440
    print(f"{a}-{b}: in={i}, out={o}, ceil(in/15)={-(-i // 15)}, floor(out/15)={o // 15}, hiệu={o // 15 - (-(-i // 15))}")

09:31-10:14: in=571, out=614, ceil(in/15)=39, floor(out/15)=40, hiệu=1
21:30-03:00: in=1290, out=1620, ceil(in/15)=86, floor(out/15)=108, hiệu=22
10:01-10:14: in=601, out=614, ceil(in/15)=41, floor(out/15)=40, hiệu=-1


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng phút

**Ý tưởng:** đi từng phút từ `in` tới `out` (đã xử lý qua nửa đêm). Một ván `[s, s + 15]` được tính khi ta "đang online" ở mốc bắt đầu `s` (tức `s ≥ in`) và vẫn online tới mốc kết thúc `s + 15` (tức `s + 15 ≤ out`). Duyệt từng phút `t`: nếu `t` là bội của 15 và `t + 15 ≤ out` thì đếm.

In [6]:
def rounds_minutes(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    if o < i:
        o += 1440
    cnt = 0
    for t in range(i, o + 1):
        if t % 15 == 0 and t + 15 <= o:
            cnt += 1
    return cnt

for (a, b), want in examples:
    print(rounds_minutes(a, b), "kỳ vọng", want)

1 kỳ vọng 1
22 kỳ vọng 22


**Độ phức tạp:** tối đa `~1440` phút ⇒ `O(1440)` — thực ra hằng số! Nhưng đây là "vét cạn theo thời gian", và nếu bài đổi sang đơn vị giây hoặc khoảng nhiều ngày thì sẽ chậm.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi các **mốc** là bội của 15 phút. Ván `k` chiếm `[15k, 15(k+1)]`. Ván `k` được chơi trọn ⇔ `15k ≥ in` và `15(k+1) ≤ out` ⇔ `k ≥ ⌈in/15⌉` và `k + 1 ≤ ⌊out/15⌋`.

⇒ số giá trị `k` hợp lệ là `⌊out/15⌋ − ⌈in/15⌉` nếu dương, ngược lại `0`.

**Quy tắc:**
- Đổi giờ sang phút; nếu `out < in` thì `out += 1440`.
- `ans = max(0, out // 15 − ceil(in / 15))`.

**Lưu ý:** `in == out` không xảy ra (đề bảo đảm), nhưng nếu xảy ra theo cách hiểu "chơi trọn 24 giờ" thì phải bàn lại — đề loại trường hợp này.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Liệt kê mọi ván (96 ván/ngày, 2 ngày)

In [7]:
def rounds_enumerate(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    if o < i:
        o += 1440
    return sum(1 for s in range(0, 2 * 1440, 15) if s >= i and s + 15 <= o)

for (a, b), want in examples:
    print(rounds_enumerate(a, b), "kỳ vọng", want)

1 kỳ vọng 1
22 kỳ vọng 22


**Độ phức tạp:** `192` ván ⇒ `O(1)`.

### 7.2 Cách 3 — Công thức làm tròn

In [8]:
def rounds_formula(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    if o < i:
        o += 1440
    return max(0, o // 15 - (i + 14) // 15)

for (a, b), want in examples:
    print(rounds_formula(a, b), "kỳ vọng", want)

1 kỳ vọng 1
22 kỳ vọng 22


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(1)` thời gian và bộ nhớ.

### 7.3 Cách 4 — Tách hai ngày (không cộng 1440)

Nếu qua nửa đêm: số ván = (ván trong `[in, 1440]`) + (ván trong `[0, out]`). Ván không thể "vắt" qua nửa đêm vì `1440` là bội của 15 — nên tách được.

In [9]:
def count_in(i, o):
    return max(0, o // 15 - -(-i // 15))

def rounds_split(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    if o >= i:
        return count_in(i, o)
    return count_in(i, 1440) + count_in(0, o)

for (a, b), want in examples:
    print(rounds_split(a, b), "kỳ vọng", want)

1 kỳ vọng 1
22 kỳ vọng 22


**Độ phức tạp:** `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ và vài trường hợp biên:

In [10]:
def trace_formula(a, b):
    i, o = to_min(a), to_min(b)
    wrap = o < i
    if wrap: o += 1440
    first = (i + 14) // 15; last = o // 15
    ans = max(0, last - first)
    print(f"{a} -> {b}: in={i}, out={o}{' (qua nửa đêm, +1440)' if wrap else ''}, "
          f"mốc đầu ceil={first} ({first * 15 // 60 % 24:02d}:{first * 15 % 60:02d}), "
          f"mốc cuối floor={last} ({last * 15 // 60 % 24:02d}:{last * 15 % 60:02d}) -> {ans} ván")
    return ans

for a, b in [("09:31", "10:14"), ("21:30", "03:00"), ("10:01", "10:14"), ("00:00", "23:59"), ("23:59", "00:00"),
             ("12:15", "12:30"), ("12:16", "12:30")]:
    trace_formula(a, b)

09:31 -> 10:14: in=571, out=614, mốc đầu ceil=39 (09:45), mốc cuối floor=40 (10:00) -> 1 ván
21:30 -> 03:00: in=1290, out=1620 (qua nửa đêm, +1440), mốc đầu ceil=86 (21:30), mốc cuối floor=108 (03:00) -> 22 ván
10:01 -> 10:14: in=601, out=614, mốc đầu ceil=41 (10:15), mốc cuối floor=40 (10:00) -> 0 ván
00:00 -> 23:59: in=0, out=1439, mốc đầu ceil=0 (00:00), mốc cuối floor=95 (23:45) -> 95 ván
23:59 -> 00:00: in=1439, out=1440 (qua nửa đêm, +1440), mốc đầu ceil=96 (00:00), mốc cuối floor=96 (00:00) -> 0 ván
12:15 -> 12:30: in=735, out=750, mốc đầu ceil=49 (12:15), mốc cuối floor=50 (12:30) -> 1 ván
12:16 -> 12:30: in=736, out=750, mốc đầu ceil=50 (12:30), mốc cuối floor=50 (12:30) -> 0 ván


Bảng tóm tắt:

| login | logout | in | out | ⌈in/15⌉ | ⌊out/15⌋ | ván |
|---|---|---|---|---|---|---|
| 09:31 | 10:14 | 571 | 614 | 39 | 40 | 1 |
| 21:30 | 03:00 | 1290 | 1620 | 86 | 108 | 22 |
| 10:01 | 10:14 | 601 | 614 | 41 | 40 | 0 (max) |
| 12:15 | 12:30 | 735 | 750 | 49 | 50 | 1 |

In [11]:
def trace_minutes(a, b):
    i, o = to_min(a), to_min(b)
    if o < i: o += 1440
    played = [(t, t + 15) for t in range(i, o + 1) if t % 15 == 0 and t + 15 <= o]
    fmt = lambda x: f"{x // 60 % 24:02d}:{x % 60:02d}"
    print(f"{a}-{b}: các ván trọn vẹn: {[fmt(s) + '-' + fmt(e) for s, e in played]}")

trace_minutes("09:31", "10:14")
trace_minutes("23:20", "00:50")

09:31-10:14: các ván trọn vẹn: ['09:45-10:00']
23:20-00:50: các ván trọn vẹn: ['23:30-23:45', '23:45-00:00', '00:00-00:15', '00:15-00:30', '00:30-00:45']


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def numberOfRounds(self, loginTime: str, logoutTime: str) -> int:
        def f(s: str) -> int:
            return int(s[:2]) * 60 + int(s[3:])
        a, b = f(loginTime), f(logoutTime)
        if b < a:
            b += 1440
        return max(0, b // 15 - (a + 14) // 15)

sol = Solution()
for (a, b), want in examples:
    got = sol.numberOfRounds(a, b)
    print(got, "OK" if got == want else "SAI")

1 OK
22 OK


## 10. Vì sao thuật toán đúng

Đặt `in`, `out` (sau khi cộng 1440 nếu qua nửa đêm, nên `in < out`). Các ván có dạng `[15k, 15k + 15]` với `k ∈ ℤ` (cả những ván của ngày hôm sau, `k ≥ 96`).

Ván `k` được chơi trọn ⇔ `in ≤ 15k` và `15k + 15 ≤ out` ⇔ `⌈in/15⌉ ≤ k ≤ ⌊out/15⌋ − 1`.

Số số nguyên trong đoạn `[A, B − 1]` (với `A = ⌈in/15⌉`, `B = ⌊out/15⌋`) là `B − A` nếu `B ≥ A`, ngược lại `0` ⇒ `max(0, B − A)`. Khi `in < out`, ta có `B − A ≥ −1` (cả hai cách nhau ít hơn một ô) — trường hợp `−1` xảy ra đúng khi `in` và `out` nằm **giữa cùng hai mốc liên tiếp**.

**Qua nửa đêm:** khoảng chơi thực là `[in, 1440) ∪ [0, out]`; ánh xạ phần thứ hai lên `[1440, out + 1440]` giữ nguyên lịch ván (vì `1440 = 96 · 15`) ⇒ cộng `1440` là hợp lệ. Cách 4 xác nhận bằng cách tách hai đoạn.

Mục 12 kiểm chứng **mọi** cặp `(login, logout)` khác nhau — toàn bộ miền input.

## 11. Phân tích Time / Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng phút | `O(1440)` | `O(1)` | dễ hiểu |
| 2. Liệt kê ván | `O(192)` | `O(1)` | |
| 3. Công thức | `O(1)` | `O(1)` | **nộp bài** |
| 4. Tách hai ngày | `O(1)` | `O(1)` | không cần +1440 |

Miền input chỉ có `1440 · 1439` cặp — có thể kiểm tra **toàn bộ** (mục 12).

In [13]:
times = [f"{h:02d}:{m:02d}" for h in range(24) for m in range(60)]
pairs = [(a, b) for a in times[::7] for b in times[::7] if a != b]
for name, fn in [("minutes", rounds_minutes), ("enumerate", rounds_enumerate), ("formula", rounds_formula),
                 ("split", rounds_split), ("submit", Solution().numberOfRounds)]:
    t0 = time.perf_counter()
    for a, b in pairs:
        fn(a, b)
    print(f"{name:9s}: {len(pairs)} cặp trong {time.perf_counter() - t0:.3f}s")

minutes  : 42230 cặp trong 0.822s


enumerate: 42230 cặp trong 0.265s
formula  : 42230 cặp trong 0.017s
split    : 42230 cặp trong 0.019s
submit   : 42230 cặp trong 0.018s


## 12. Bộ test

Kiểm tra **mọi** cặp thời điểm khác nhau (`1440 × 1439 ≈ 2.07 triệu` cặp) cho công thức, đối chiếu với Cách 2 (liệt kê ván). Mô phỏng từng phút chậm hơn nên chỉ đối chiếu trên một mẫu.

In [14]:
fns = [rounds_minutes, rounds_enumerate, rounds_formula, rounds_split, lambda a, b: Solution().numberOfRounds(a, b)]

for (a, b), want in examples:
    for fn in fns:
        assert fn(a, b) == want

edge = [(("10:01", "10:14"), 0), (("12:15", "12:30"), 1), (("12:16", "12:30"), 0), (("12:15", "12:29"), 0),
        (("00:00", "23:59"), 95), (("00:00", "23:45"), 95), (("23:59", "00:00"), 0), (("23:45", "00:00"), 1),
        (("00:01", "00:00"), 95), (("20:00", "06:00"), 40)]
for (a, b), want in edge:
    for fn in fns:
        assert fn(a, b) == want, (fn, a, b)

# toàn bộ miền input cho các cách O(1): dùng phút trực tiếp cho nhanh
def formula_min(i, o):
    if o < i: o += 1440
    return max(0, o // 15 - (i + 14) // 15)
def split_min(i, o):
    if o >= i: return count_in(i, o)
    return count_in(i, 1440) + count_in(0, o)
def enum_min(i, o):
    if o < i: o += 1440
    # số s là bội của 15 với i <= s và s + 15 <= o
    lo = (i + 14) // 15; hi = (o - 15) // 15
    return max(0, hi - lo + 1)

cnt = 0
for i in range(1440):
    for o in range(1440):
        if i != o:
            a = formula_min(i, o)
            assert a == split_min(i, o) == enum_min(i, o)
            cnt += 1
print(f"đã kiểm {cnt} cặp (toàn bộ miền input): công thức = tách hai ngày = đếm bội số")

đã kiểm 2072160 cặp (toàn bộ miền input): công thức = tách hai ngày = đếm bội số


In [15]:
rng = random.Random(2024)
for _ in range(3000):
    a, b = rng.sample(times, 2)
    want = rounds_minutes(a, b)
    for fn in fns[1:]:
        assert fn(a, b) == want, (fn, a, b)
print("3000 cặp ngẫu nhiên: mọi cách khớp mô phỏng từng phút")

3000 cặp ngẫu nhiên: mọi cách khớp mô phỏng từng phút


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên `max(0, …)` (cùng một ô 15 phút cho `−1`)

In [16]:
def wrong_no_max(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    if o < i: o += 1440
    return o // 15 - (i + 14) // 15                      # SAI: có thể âm

for a, b in [("10:01", "10:14"), ("12:16", "12:29")]:
    print(f"{a}-{b}: sai = {wrong_no_max(a, b)} | đúng = {Solution().numberOfRounds(a, b)}")

10:01-10:14: sai = -1 | đúng = 0
12:16-12:29: sai = -1 | đúng = 0


**Sửa:** `max(0, …)`.

### Lỗi 2 — Làm tròn sai chiều (`⌊in/15⌋` thay vì `⌈in/15⌉`)

In [17]:
def wrong_floor_in(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    if o < i: o += 1440
    return max(0, o // 15 - i // 15)                     # SAI: tính cả ván đã bắt đầu trước khi vào

for (a, b), want in examples:
    print(f"{a}-{b}: sai = {wrong_floor_in(a, b)} | đúng = {want}")

09:31-10:14: sai = 2 | đúng = 1
21:30-03:00: sai = 22 | đúng = 22


Ví dụ 1: tính nhầm ván `09:30–09:45` dù vào lúc `09:31`. **Sửa:** ván đầu tiên bắt đầu ở mốc **≥ in** ⇒ làm tròn **lên**.

### Lỗi 3 — Tính trần bằng `i // 15 + 1` (sai khi `in` là bội của 15)

In [18]:
def wrong_ceil(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    if o < i: o += 1440
    return max(0, o // 15 - (i // 15 + 1))               # SAI khi i % 15 == 0

for a, b in [("12:15", "12:30"), ("00:00", "23:59"), ("09:31", "10:14")]:
    print(f"{a}-{b}: sai = {wrong_ceil(a, b)} | đúng = {Solution().numberOfRounds(a, b)}")

12:15-12:30: sai = 0 | đúng = 1
00:00-23:59: sai = 94 | đúng = 95
09:31-10:14: sai = 1 | đúng = 1


Đăng nhập **đúng lúc** ván bắt đầu (`12:15`) vẫn chơi trọn ván đó. **Sửa:** `(i + 14) // 15` hoặc `-(-i // 15)`.

### Lỗi 4 — Không xử lý qua nửa đêm (hoặc cộng 1440 vào `login`)

In [19]:
def wrong_no_wrap(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    return max(0, o // 15 - (i + 14) // 15)             # SAI: o < i cho kết quả 0

def wrong_wrap_login(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    if o < i: i += 1440                                  # SAI: cộng nhầm vào login
    return max(0, o // 15 - (i + 14) // 15)

a, b = "21:30", "03:00"
print("không xử lý:", wrong_no_wrap(a, b), "| cộng vào login:", wrong_wrap_login(a, b), "| đúng:", Solution().numberOfRounds(a, b))

không xử lý: 0 | cộng vào login: 0 | đúng: 22


**Sửa:** khi `out < in`, cộng `1440` vào **out**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài thời gian dạng chuỗi ⇒ đổi sang **phút từ nửa đêm** ngay.
- Khoảng vòng qua nửa đêm ⇒ cộng **một ngày** vào điểm kết thúc (hoặc tách hai đoạn).
- "Số khối độ dài `L` nằm trọn trong `[a, b]`" ⇒ `max(0, ⌊b/L⌋ − ⌈a/L⌉)` với khối bắt đầu ở bội của `L`.
- Miền input nhỏ ⇒ **vét cạn toàn bộ** để kiểm chứng công thức (rất đáng làm với bài làm tròn).
- Họ bài: 539 *Minimum Time Difference*, 1360 *Number of Days Between Two Dates*, 2224 *Minimum Number of Operations to Convert Time*.

## 15. Biến thể của bài

1. **Độ dài ván `L` tùy ý** (chia hết 1440): thay `15` bằng `L`.
2. **Số ván chơi được ít nhất một phần** (giao khác rỗng với khoảng chơi): `⌈out/L⌉ − ⌊in/L⌋` (cẩn thận biên chạm điểm).
3. **Nhiều phiên đăng nhập** (các khoảng có thể chồng nhau): hợp các khoảng trước rồi đếm từng khoảng.
4. **Khoảng kéo dài nhiều ngày** (có ngày tháng): đổi sang phút tuyệt đối, công thức giữ nguyên.

In [20]:
def rounds_len(i, o, L):
    """Biến thể 1."""
    if o < i: o += 1440
    return max(0, o // L - -(-i // L))

def rounds_touched(i, o, L=15):
    """Biến thể 2: số ván có ít nhất một phút chung (khoảng mở của ván (s, s+L) giao (i, o))."""
    if o < i: o += 1440
    return max(0, -(-o // L) - i // L)

def rounds_touched_brute(i, o, L=15):
    if o < i: o += 1440
    return sum(1 for s in range(0, 2 * 1440, L) if max(s, i) < min(s + L, o))

for L in [5, 10, 15, 30, 60]:
    for _ in range(300):
        i, o = random.Random(L * 1000 + _).sample(range(1440), 2)
        brute = sum(1 for s in range(0, 2 * 1440, L) if s >= i and s + L <= (o + 1440 if o < i else o))
        assert rounds_len(i, o, L) == brute
for _ in range(2000):
    i, o = random.Random(_).sample(range(1440), 2)
    assert rounds_touched(i, o) == rounds_touched_brute(i, o)
print("biến thể 1 (độ dài L) và 2 (ván chạm khoảng chơi) khớp vét cạn")

biến thể 1 (độ dài L) và 2 (ván chạm khoảng chơi) khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng phút | đi từng phút | `O(1440)` | `O(1)` | kiểm chứng |
| 2. Liệt kê ván | 192 ván | `O(1)` | `O(1)` | kiểm chứng |
| 3. Công thức | `⌊out/15⌋ − ⌈in/15⌉` | `O(1)` | `O(1)` | **nộp bài** |
| 4. Tách hai ngày | cộng hai đoạn | `O(1)` | `O(1)` | khi ngại +1440 |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `numberOfRounds("00:47", "00:57")`, `numberOfRounds("00:00", "00:15")`, `numberOfRounds("23:46", "00:01")`?

**Bài 2 (tính tay):** `"18:51"` → `"04:54"`: tính `in`, `out`, mốc đầu, mốc cuối, đáp án.

**Bài 3 (điền TODO):** hoàn thiện `played_rounds(loginTime, logoutTime)` trả về **danh sách** chuỗi `"hh:mm"` là giờ bắt đầu các ván trọn vẹn.

**Bài 4 (tự viết lại):** viết lại lời giải **không dùng `split`/`int` trên từng phần** mà dùng `ord` để đổi chuỗi giờ sang phút.

**Bài 5 (mini challenge):** biến thể 3 — cho danh sách nhiều phiên `[(login, logout), …]` (có thể chồng nhau, có thể qua nửa đêm, mỗi phiên trong cùng một "ngày chơi" giả định kéo dài tối đa 48 giờ tính từ `00:00` ngày 1). Đếm số ván **phân biệt** được chơi trọn trong ít nhất một phiên. (Biến thể tự đặt.)

In [21]:
def played_rounds(loginTime, logoutTime):
    # TODO: liệt kê k từ ceil(in/15) tới floor(out/15) - 1, đổi 15k về "hh:mm" (mod 24h)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `00:47–00:57` nằm giữa hai mốc `00:45` và `01:00` ⇒ `0`. (2) `00:00–00:15` ⇒ đúng một ván ⇒ `1`. (3) `23:46 → 00:01`: mốc đầu `00:00` (ngày sau), mốc cuối `00:00` ⇒ `0`.

**Bài 2:** (1) `in = 1131`, `out = 294 + 1440 = 1734`. (2) `⌈1131/15⌉ = 76`, `⌊1734/15⌋ = 115`. (3) Đáp án `39`.

**Bài 3:** (1) Tính `A`, `B` như Cách 3. (2) Với `k ∈ [A, B)`, giờ bắt đầu `15k mod 1440`. (3) Định dạng `f"{m // 60:02d}:{m % 60:02d}"`.

**Bài 4:** (1) `h = (ord(s[0]) − 48) · 10 + ord(s[1]) − 48`. (2) Tương tự phút. (3) So với `to_min` trên mọi chuỗi giờ.

**Bài 5:** (1) Đổi mỗi phiên sang đoạn `[i, o]` với `o += 1440` nếu qua nửa đêm. (2) Tập các `k` (chỉ số ván) chơi trọn của mỗi phiên là một đoạn số nguyên. (3) Hợp các đoạn số nguyên rồi đếm (hoặc dùng `set` vì tối đa 192 ván).

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert [Solution().numberOfRounds("00:47", "00:57"), Solution().numberOfRounds("00:00", "00:15"),
        Solution().numberOfRounds("23:46", "00:01")] == [0, 1, 0]

# Bài 2
assert Solution().numberOfRounds("18:51", "04:54") == 39

# Bài 3
def played_rounds_ans(loginTime, logoutTime):
    i, o = to_min(loginTime), to_min(logoutTime)
    if o < i: o += 1440
    A, B = (i + 14) // 15, o // 15
    return [f"{15 * k % 1440 // 60:02d}:{15 * k % 60:02d}" for k in range(A, B)]
print(played_rounds_ans("09:31", "10:14"), played_rounds_ans("23:20", "00:50"))

# Bài 4
def to_min_ord(s):
    return ((ord(s[0]) - 48) * 10 + ord(s[1]) - 48) * 60 + (ord(s[3]) - 48) * 10 + ord(s[4]) - 48

assert all(to_min_ord(t) == to_min(t) for t in times)
for (a, b), want in examples:
    assert len(played_rounds_ans(a, b)) == want
print("Bài 1-4: OK")

['09:45'] ['23:30', '23:45', '00:00', '00:15', '00:30']
Bài 1-4: OK


In [23]:
# Bài 5 — nhiều phiên, đếm ván phân biệt
def distinct_rounds(sessions):
    ks = set()
    for a, b in sessions:
        i, o = to_min(a), to_min(b)
        if o < i: o += 1440
        ks.update(range((i + 14) // 15, o // 15))
    return len(ks)

def distinct_rounds_brute(sessions):
    covered = [False] * (2 * 1440 + 1)
    for a, b in sessions:
        i, o = to_min(a), to_min(b)
        if o < i: o += 1440
        for t in range(i, o + 1): covered[t] = True
    # ván k chơi trọn trong một phiên: cần một phiên chứa cả [15k, 15k+15]
    cnt = 0
    for k in range(0, 2 * 96):
        s, e = 15 * k, 15 * k + 15
        ok = False
        for a, b in sessions:
            i, o = to_min(a), to_min(b)
            if o < i: o += 1440
            if i <= s and e <= o: ok = True; break
        cnt += ok
    return cnt

rng = random.Random(271)
for _ in range(300):
    sessions = [tuple(rng.sample(times, 2)) for _ in range(rng.randint(1, 4))]
    assert distinct_rounds(sessions) == distinct_rounds_brute(sessions)
print("Bài 5: OK;", distinct_rounds([("09:31", "10:14"), ("09:50", "11:00"), ("21:30", "03:00")]))

Bài 5: OK; 27


## 20. Template nhớ nhanh + Checklist

```python
def f(s): return int(s[:2]) * 60 + int(s[3:])
a, b = f(loginTime), f(logoutTime)
if b < a:
    b += 1440                               # qua nửa đêm: cộng một ngày vào thời điểm RA
return max(0, b // 15 - (a + 14) // 15)     # floor(out/15) - ceil(in/15), không âm
```

**Checklist trước khi submit:**
- [ ] Đăng nhập: làm tròn **lên** (`(a + 14) // 15`), đăng xuất: làm tròn **xuống**.
- [ ] `max(0, …)` cho trường hợp cùng một ô 15 phút.
- [ ] Qua nửa đêm: cộng `1440` vào **out**, không phải in.
- [ ] Đăng nhập đúng mốc (`12:15`) vẫn chơi ván đó.
- [ ] Kiểm tra `"23:45" → "00:00"` (1 ván) và `"23:59" → "00:00"` (0 ván).